New renewable generation and land requirements (2030–2050)




In [ ]:
# STEP 1 — Import libraries and define the editable Windows paths.
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# Edit only these paths if the project or workbook is moved.
PROJECT_ROOT = Path(os.environ.get("RESEARCH_ANALYSIS_ROOT", r"Z:\Research-analysis"))
NOTEBOOK_NAME = "F1-v2-3"
INPUT_WORKBOOK = Path(
    os.environ.get(
        "F1_V2_3_INPUT_WORKBOOK",
        str(PROJECT_ROOT / "outputs" / "figures" / "F1-v2-1" / "F1-v2-1_data.xlsx"),
    )
)
OUTPUT_FOLDER = Path(
    os.environ.get(
        "F1_V2_3_OUTPUT_FOLDER",
        str(PROJECT_ROOT / "outputs" / "figures" / NOTEBOOK_NAME),
    )
)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Only these five years are loaded and plotted.
YEARS = [2030, 2035, 2040, 2045, 2050]
FIGURE_RUNS = ["Run_G0001", "Run_G0009"]
FLEXIBLE_RUNS = ["Run_G0001", "Run_G0009"]

print("Input workbook:", INPUT_WORKBOOK)
print("Output folder :", OUTPUT_FOLDER)
print("Plot years    :", YEARS)


In [ ]:
# STEP 2 — Load the existing Figure_Data sheet and keep only 2030–2050.
if not INPUT_WORKBOOK.is_file():
    raise FileNotFoundError(
        f"Excel data workbook not found: {INPUT_WORKBOOK}\n"
        "Check that the Z: drive is connected, or edit INPUT_WORKBOOK in STEP 1."
    )

# Match the sheet name without depending on capitalization or surrounding spaces.
excel_file = pd.ExcelFile(INPUT_WORKBOOK)
sheet_lookup = {str(name).strip().casefold(): name for name in excel_file.sheet_names}
figure_sheet = sheet_lookup.get("figure_data")
if figure_sheet is None:
    raise KeyError(
        f"The workbook does not contain a Figure_Data sheet. Found: {excel_file.sheet_names}"
    )

figure_data = pd.read_excel(INPUT_WORKBOOK, sheet_name=figure_sheet)
figure_data.columns = [str(column).strip() for column in figure_data.columns]

required_columns = [
    "Run", "Year",
    "Total Renewables Target (TWh)",
    "New Solar (TWh)", "New Wind (TWh)",
    "Existing Solar (MWh)", "Existing Wind (MWh)",
    "New Solar Area (Mha)", "New Wind Area (Mha)",
    "Existing Solar Area (ha)", "Existing Wind Area (ha)",
]
missing_columns = [column for column in required_columns if column not in figure_data.columns]
if missing_columns:
    raise KeyError(
        "Figure_Data is missing required columns:\n  " + "\n  ".join(missing_columns)
    )

# Normalize identifiers before filtering so Excel text/number formatting cannot alter selection.
figure_data["Run"] = figure_data["Run"].astype(str).str.strip()
figure_data["Year"] = pd.to_numeric(figure_data["Year"], errors="coerce")
figure_data = figure_data.loc[
    figure_data["Run"].isin(FIGURE_RUNS) & figure_data["Year"].isin(YEARS)
].copy()
figure_data["Year"] = figure_data["Year"].astype(int)

numeric_columns = [column for column in required_columns if column not in {"Run", "Year"}]
for column in numeric_columns:
    figure_data[column] = pd.to_numeric(figure_data[column], errors="coerce")
if figure_data[numeric_columns].isna().any().any():
    bad_columns = figure_data[numeric_columns].columns[figure_data[numeric_columns].isna().any()].tolist()
    raise ValueError(f"Non-numeric or missing figure values were found in: {bad_columns}")

# Recalculate totals from their components rather than trusting cached Excel totals.
figure_data["New Generation Total (TWh)"] = (
    figure_data["New Solar (TWh)"] + figure_data["New Wind (TWh)"]
)
figure_data["New Renewable Area Total (Mha)"] = (
    figure_data["New Solar Area (Mha)"] + figure_data["New Wind Area (Mha)"]
)
figure_data = figure_data.sort_values(["Run", "Year"]).reset_index(drop=True)

print(f"Loaded {len(figure_data):,} figure rows from sheet: {figure_sheet}")
display(figure_data.head(10))


In [ ]:
# STEP 3 — Validate that every required run has all five requested years exactly once.
if figure_data.duplicated(["Run", "Year"]).any():
    duplicates = figure_data.loc[
        figure_data.duplicated(["Run", "Year"], keep=False), ["Run", "Year"]
    ]
    raise AssertionError(f"Duplicate run/year rows found:\n{duplicates}")

expected_pairs = pd.MultiIndex.from_product([FIGURE_RUNS, YEARS], names=["Run", "Year"])
found_pairs = pd.MultiIndex.from_frame(figure_data[["Run", "Year"]])
missing_pairs = expected_pairs.difference(found_pairs)
unexpected_pairs = found_pairs.difference(expected_pairs)
if len(missing_pairs) or len(unexpected_pairs):
    raise AssertionError(
        "Figure data completeness check failed.\n"
        f"Missing: {list(missing_pairs)}\n"
        f"Unexpected: {list(unexpected_pairs)}"
    )

if (figure_data[numeric_columns] < 0).any().any():
    negative_columns = figure_data[numeric_columns].columns[
        (figure_data[numeric_columns] < 0).any()
    ].tolist()
    raise AssertionError(f"Negative values found in columns expected to be non-negative: {negative_columns}")

print("INPUT DATA VALIDATION PASSED")
print("Runs :", ", ".join(FIGURE_RUNS))
print("Years:", ", ".join(map(str, YEARS)))


In [4]:
# STEP 4 — Editable layout, spacing, colors, and legend positions.
# Scenario colors distinguish the two energy pathways plotted under the same
# Flexible climate and Baseline biodiversity configuration.
SCENARIO_COLORS = {
    "Step Change": "#56B4E9",
    "Accelerated Transition": "#E6AB02",
}
EXISTING_COLOR = "#8C8C8C"
GRID_COLOR = "#D9D9D9"

FIGURE_1_SIZE = (11.0, 10.2)
FIGURE_2_SIZE = (14.5, 10.2)
FIGURE_3_SIZE = (11.0, 10.2)
FIGURE_4_SIZE = (11.0, 10.2)
FIGURE_DPI = 400

BAR_WIDTH = 0.32
SOLAR_ALPHA = 1.00
WIND_ALPHA = 0.45

# Figure 4 spacing controls.
STACKED_LEFT = 0.11
STACKED_RIGHT = 0.89
STACKED_TOP = 0.88
STACKED_BOTTOM = 0.24
STACKED_HSPACE = 0.18
STACKED_TITLE_Y = 0.975
STACKED_LEGEND_LINE1_Y = 0.095
STACKED_LEGEND_LINE2_Y = 0.052

# Figure 2 spacing and shared-label controls.
FIG2_LEFT = 0.09
FIG2_RIGHT = 0.91
FIG2_TOP = 0.87
FIG2_BOTTOM = 0.23
FIG2_WSPACE = 0.25
FIG2_HSPACE = 0.25
FIG2_TITLE_Y = 0.975
FIG2_LEGEND_LINE1_Y = 0.095
FIG2_LEGEND_LINE2_Y = 0.052
FIG2_SHARED_LEFT_X = 0.018
FIG2_SHARED_RIGHT_X = 0.982
FIG2_GENERATION_LABEL_Y = 0.64
FIG2_AREA_LABEL_Y = 0.32


def scenario_name(run_name):
    """Map the four plotted runs to their energy scenario."""
    return "Step Change" if run_name in {"Run_G0001", "Run_G0002"} else "Accelerated Transition"


def scenario_color(run_name):
    return SCENARIO_COLORS[scenario_name(run_name)]


def run_frame(run_names):
    """Return complete, ordered figure data for the requested runs."""
    frame = figure_data.loc[figure_data["Run"].isin(run_names)].copy()
    expected = pd.MultiIndex.from_product([run_names, YEARS], names=["Run", "Year"])
    found = pd.MultiIndex.from_frame(frame[["Run", "Year"]])
    missing = expected.difference(found)
    if len(missing):
        raise ValueError(f"Missing figure data for: {list(missing)}")
    return frame.set_index(["Run", "Year"]).reindex(expected)


def grouped_offsets(number_of_runs):
    """Center grouped scenario bars around each year tick."""
    if number_of_runs == 1:
        return np.array([0.0])
    return np.linspace(
        -BAR_WIDTH * (number_of_runs - 1) / 2,
        BAR_WIDTH * (number_of_runs - 1) / 2,
        number_of_runs,
    )

In [5]:
# STEP 5 — Plotting and two-line legend helpers.
def draw_generation_panel(
    axis,
    run_names,
    title=None,
    show_left_label=True,
    show_right_label=True,
    show_x_labels=True,
    generation_limit=None,
    target_limit=None,
    include_existing=False,
    show_target=True,
):
    """Draw generation bars, optionally with grey existing capacity and a target line."""
    data = run_frame(run_names)
    target_axis = axis.twinx() if show_target else None
    x_base = np.arange(len(YEARS))
    offsets = grouped_offsets(len(run_names))

    for offset, run_name in zip(offsets, run_names):
        run_data = data.loc[run_name]
        x_values = x_base + offset
        color = scenario_color(run_name)
        solar = run_data["New Solar (TWh)"].to_numpy()
        wind = run_data["New Wind (TWh)"].to_numpy()
        existing = (
            run_data["Existing Solar (MWh)"].to_numpy()
            + run_data["Existing Wind (MWh)"].to_numpy()
        ) / 1_000_000 if include_existing else np.zeros(len(run_data))

        if include_existing:
            axis.bar(
                x_values, existing, width=BAR_WIDTH,
                color=EXISTING_COLOR, edgecolor=EXISTING_COLOR,
                linewidth=0.7, zorder=3,
            )
        axis.bar(
            x_values, solar, width=BAR_WIDTH, bottom=existing,
            color=color, alpha=SOLAR_ALPHA, edgecolor=color,
            linewidth=0.7, zorder=3,
        )
        axis.bar(
            x_values, wind, width=BAR_WIDTH, bottom=existing + solar,
            color=color, alpha=WIND_ALPHA, edgecolor=color,
            linewidth=0.7, zorder=3,
        )

        if show_target:
            target_axis.plot(
                x_values,
                run_data["Total Renewables Target (TWh)"].to_numpy(),
                color=color, linestyle="--", marker="o",
                linewidth=1.8, markersize=4.8,
                markeredgecolor="white", markeredgewidth=0.6,
                zorder=5,
            )

    axis.set_xticks(x_base)
    axis.set_xticklabels(YEARS if show_x_labels else [])
    if show_x_labels:
        axis.set_xlabel("Year")
    if show_left_label:
        axis.set_ylabel("Solar and Wind generation (TWh)")
    if show_target and show_right_label:
        target_axis.set_ylabel("Total renewables target (TWh)")
    if title:
        axis.set_title(title, loc="center", fontsize=11.5, fontweight="bold", pad=10)
    axis.grid(axis="y", color=GRID_COLOR, linewidth=0.65)
    axis.grid(axis="x", visible=False)
    axis.set_axisbelow(True)
    axis.set_ylim(bottom=0, top=generation_limit)
    if show_target:
        target_axis.grid(False)
        target_axis.set_ylim(bottom=0, top=target_limit)
    return target_axis


def draw_area_panel(
    axis,
    run_names,
    title=None,
    show_y_label=True,
    show_x_labels=True,
    area_limit=None,
    include_existing=False,
):
    """Draw new-renewables land area, optionally on a grey existing-capacity base."""
    data = run_frame(run_names)
    x_base = np.arange(len(YEARS))
    offsets = grouped_offsets(len(run_names))

    for offset, run_name in zip(offsets, run_names):
        run_data = data.loc[run_name]
        x_values = x_base + offset
        color = scenario_color(run_name)
        solar = run_data["New Solar Area (Mha)"].to_numpy()
        wind = run_data["New Wind Area (Mha)"].to_numpy()
        existing = (
            run_data["Existing Solar Area (ha)"].to_numpy()
            + run_data["Existing Wind Area (ha)"].to_numpy()
        ) / 1_000_000 if include_existing else np.zeros(len(run_data))

        if include_existing:
            axis.bar(
                x_values, existing, width=BAR_WIDTH,
                color=EXISTING_COLOR, edgecolor=EXISTING_COLOR,
                linewidth=0.7, zorder=3,
            )
        axis.bar(
            x_values, solar, width=BAR_WIDTH, bottom=existing,
            color=color, alpha=SOLAR_ALPHA, edgecolor=color,
            linewidth=0.7, zorder=3,
        )
        axis.bar(
            x_values, wind, width=BAR_WIDTH, bottom=existing + solar,
            color=color, alpha=WIND_ALPHA, edgecolor=color,
            linewidth=0.7, zorder=3,
        )

    axis.set_xticks(x_base)
    axis.set_xticklabels(YEARS if show_x_labels else [])
    if show_x_labels:
        axis.set_xlabel("Year")
    if show_y_label:
        axis.set_ylabel("Solar and Wind land area (Mha)")
    if title:
        axis.set_title(title, loc="center", fontsize=11.5, fontweight="bold", pad=10)
    axis.grid(axis="y", color=GRID_COLOR, linewidth=0.65)
    axis.grid(axis="x", visible=False)
    axis.set_axisbelow(True)
    axis.set_ylim(bottom=0, top=area_limit)


def technology_handles(show_target=True, include_existing=False):
    """First legend line: technologies and the optional target/existing items."""
    handles = []
    if include_existing:
        handles.append(Patch(facecolor=EXISTING_COLOR, edgecolor=EXISTING_COLOR, label="Existing capacity"))
    handles.extend(
        [
            Patch(facecolor="#666666", edgecolor="#666666", alpha=SOLAR_ALPHA, label="Utility Solar generation"),
            Patch(facecolor="#666666", edgecolor="#666666", alpha=WIND_ALPHA, label="Onshore Wind"),
        ]
    )
    if show_target:
        handles.append(
            Line2D([0], [0], color="#333333", linestyle="--", marker="o", label="Total renewables target")
        )
    return handles


def scenario_handles():
    """Second legend line: the two energy scenarios."""
    return [
        Line2D([0], [0], color=SCENARIO_COLORS["Step Change"], linewidth=6, label="Step Change"),
        Line2D([0], [0], color=SCENARIO_COLORS["Accelerated Transition"], linewidth=6, label="Accelerated Transition"),
    ]


def add_two_line_legend(
    figure,
    show_target=True,
    include_existing=False,
    line1_y=STACKED_LEGEND_LINE1_Y,
    line2_y=STACKED_LEGEND_LINE2_Y,
):
    """Add two separately editable legend rows below a figure."""
    first_line = technology_handles(show_target=show_target, include_existing=include_existing)
    figure.legend(
        handles=first_line, loc="lower center", bbox_to_anchor=(0.5, line1_y),
        ncol=len(first_line), frameon=False, fontsize=9.0,
    )
    figure.legend(
        handles=scenario_handles(), loc="lower center", bbox_to_anchor=(0.5, line2_y),
        ncol=2, frameon=False, fontsize=9.0,
    )

In [ ]:
# STEP 6 — Single output figure: Figure 4 design using Run 1 and Run 9.
flexible_data = run_frame(FLEXIBLE_RUNS)
generation_limit = flexible_data["New Generation Total (TWh)"].max() * 1.12
area_limit = flexible_data["New Renewable Area Total (Mha)"].max() * 1.12

fig4, axes4 = plt.subplots(2, 1, figsize=FIGURE_4_SIZE, sharex=True)
draw_generation_panel(
    axes4[0], FLEXIBLE_RUNS,
    title="New renewable generation",
    show_right_label=False, show_x_labels=False,
    generation_limit=generation_limit,
    include_existing=False,
    show_target=False,
)
draw_area_panel(
    axes4[1], FLEXIBLE_RUNS,
    title="Land required for new renewables",
    area_limit=area_limit,
    include_existing=False,
)
fig4.suptitle(
    "Flexible pathway new renewable generation and land requirements, 2030–2050",
    fontsize=14, fontweight="bold", y=STACKED_TITLE_Y,
)
add_two_line_legend(fig4, show_target=False, include_existing=False)
fig4.subplots_adjust(
    left=STACKED_LEFT, right=STACKED_RIGHT,
    top=STACKED_TOP, bottom=STACKED_BOTTOM,
    hspace=STACKED_HSPACE,
)

figure4_base = OUTPUT_FOLDER / "F1-v2-3_Figure4"
fig4.savefig(figure4_base.with_suffix(".png"), dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
fig4.savefig(figure4_base.with_suffix(".jpg"), dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
fig4.savefig(figure4_base.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
plt.show()
print("Figure 4 saved:", figure4_base)

In [ ]:
# STEP 7 — Final output validation for the single requested figure.
expected_files = [
    OUTPUT_FOLDER / "F1-v2-3_Figure4.png",
    OUTPUT_FOLDER / "F1-v2-3_Figure4.jpg",
    OUTPUT_FOLDER / "F1-v2-3_Figure4.pdf",
]

missing_outputs = [
    path
    for path in expected_files
    if not path.is_file()
]

if missing_outputs:
    raise FileNotFoundError(
        "The following expected outputs were not created:\n"
        + "\n".join(
            f"  {path}"
            for path in missing_outputs
        )
    )

print("ALL OUTPUT VALIDATIONS PASSED")
print("Runs plotted: Run_G0001 and Run_G0009")
for path in expected_files:
    print(" ", path)